<div align="center">

# 🦙✨ Ollama + Open WebUI on Google Colab  
### **Masterpiece Edition**

Run a GGUF model on a Colab GPU with a polished Open WebUI frontend, clean status panels, reusable helpers, and Pinggy share links.

<br>

<table>
<tr>
<td><b>Runtime</b></td>
<td>Google Colab GPU</td>
</tr>
<tr>
<td><b>Stack</b></td>
<td>Ollama · Open WebUI · Pinggy · GGUF</td>
</tr>
<tr>
<td><b>Flow</b></td>
<td>Install → Serve → Download → Register → Tunnel → Launch</td>
</tr>
</table>

</div>

---

## 🧭 How to run this notebook

Run the cells **from top to bottom**. Wait for each cell to finish and show a success message before moving to the next one.

> **Important:** Keep the final Open WebUI cell running. Closing or stopping it will stop the web server.

> **Security note:** Tunnel links can expose services outside Colab. Share them only with people you trust, and stop the runtime when you are finished.


In [ ]:
#@title 🎨 Masterpiece UI Toolkit
#@markdown Beautiful status cards, diagnostics, and helper functions used by the rest of the notebook.

from IPython.display import display, HTML
from pathlib import Path
import html
import json
import os
import shlex
import socket
import subprocess
import textwrap
import time
import urllib.request
import urllib.error

STYLE = """
<style>
:root {
  --bg: #0f1020;
  --panel: rgba(255,255,255,.08);
  --panel-2: rgba(255,255,255,.12);
  --line: rgba(255,255,255,.18);
  --text: #f7f7ff;
  --muted: #b9badb;
  --green: #33d17a;
  --red: #ff6b6b;
  --blue: #79c0ff;
  --gold: #ffd166;
  --purple: #c084fc;
}
.master-wrap {
  font-family: Inter, ui-sans-serif, system-ui, -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
  color: var(--text);
  background:
    radial-gradient(circle at top left, rgba(192,132,252,.32), transparent 28%),
    radial-gradient(circle at top right, rgba(121,192,255,.28), transparent 30%),
    linear-gradient(135deg, #0f1020 0%, #17172f 52%, #101827 100%);
  border: 1px solid var(--line);
  border-radius: 24px;
  padding: 22px 24px;
  margin: 14px 0;
  box-shadow: 0 24px 70px rgba(0,0,0,.30);
}
.master-title {
  font-size: 24px;
  font-weight: 800;
  letter-spacing: -.02em;
  margin: 0 0 6px 0;
}
.master-subtitle {
  color: var(--muted);
  font-size: 14px;
  margin: 0;
}
.master-grid {
  display: grid;
  grid-template-columns: repeat(auto-fit, minmax(190px, 1fr));
  gap: 12px;
  margin-top: 16px;
}
.master-card {
  background: var(--panel);
  border: 1px solid var(--line);
  border-radius: 18px;
  padding: 15px;
}
.master-kicker {
  color: var(--muted);
  text-transform: uppercase;
  font-size: 11px;
  letter-spacing: .11em;
  margin-bottom: 7px;
}
.master-value {
  font-size: 16px;
  font-weight: 750;
  overflow-wrap: anywhere;
}
.master-badge {
  display: inline-flex;
  align-items: center;
  gap: 8px;
  padding: 8px 11px;
  border-radius: 999px;
  background: var(--panel-2);
  border: 1px solid var(--line);
  font-weight: 750;
  margin-top: 12px;
}
.master-ok { color: var(--green); }
.master-bad { color: var(--red); }
.master-info { color: var(--blue); }
.master-warn { color: var(--gold); }
.master-log {
  white-space: pre-wrap;
  color: #e8e8ff;
  background: rgba(0,0,0,.32);
  border: 1px solid rgba(255,255,255,.10);
  border-radius: 14px;
  padding: 12px;
  max-height: 330px;
  overflow: auto;
  margin-top: 10px;
}
.master-button {
  display: inline-block;
  padding: 12px 15px;
  margin: 8px 8px 0 0;
  border-radius: 14px;
  color: white !important;
  text-decoration: none !important;
  font-weight: 800;
  border: 1px solid rgba(255,255,255,.22);
  background: linear-gradient(135deg, #7c3aed, #2563eb);
  box-shadow: 0 10px 28px rgba(37,99,235,.25);
}
</style>
"""

display(HTML(STYLE + """
<div class="master-wrap">
  <div class="master-title">✨ Interface loaded</div>
  <p class="master-subtitle">Your notebook now has a polished control-panel look with clean success, warning, and log cards.</p>
</div>
"""))

def _escape(value):
    return html.escape(str(value), quote=True)

def panel(title, subtitle="", cards=None, badge=None, badge_class="master-info"):
    cards = cards or []
    card_html = "".join(
        f"""
        <div class="master-card">
          <div class="master-kicker">{_escape(k)}</div>
          <div class="master-value">{_escape(v)}</div>
        </div>
        """
        for k, v in cards
    )
    badge_html = f'<div class="master-badge {badge_class}">{badge}</div>' if badge else ""
    display(HTML(STYLE + f"""
    <div class="master-wrap">
      <div class="master-title">{title}</div>
      <p class="master-subtitle">{subtitle}</p>
      <div class="master-grid">{card_html}</div>
      {badge_html}
    </div>
    """))

def success(msg):
    display(HTML(STYLE + f"""
    <div class="master-wrap">
      <div class="master-badge master-ok">✅ {_escape(msg)}</div>
    </div>
    """))

def failure(msg):
    display(HTML(STYLE + f"""
    <div class="master-wrap">
      <div class="master-badge master-bad">❌ {_escape(msg)}</div>
    </div>
    """))

def info(msg):
    display(HTML(STYLE + f"""
    <div class="master-wrap">
      <div class="master-badge master-info">🔗 {_escape(msg)}</div>
    </div>
    """))

def warn(msg):
    display(HTML(STYLE + f"""
    <div class="master-wrap">
      <div class="master-badge master-warn">⚠️ {_escape(msg)}</div>
    </div>
    """))

def run_shell(label, command, timeout=None, env=None):
    """Run a shell command and render a beautiful result card."""
    started = time.time()
    display(HTML(STYLE + f"""
    <div class="master-wrap">
      <div class="master-title">⏳ {label}</div>
      <p class="master-subtitle"><code>{_escape(command)}</code></p>
    </div>
    """))
    try:
        result = subprocess.run(
            ["bash", "-lc", command],
            capture_output=True,
            text=True,
            timeout=timeout,
            env=env,
        )
    except subprocess.TimeoutExpired as e:
        logs = (e.stdout or "") + "\n" + (e.stderr or "")
        failure(f"{label} timed out.")
        if logs.strip():
            display_log(label, logs)
        raise

    elapsed = f"{time.time() - started:.1f}s"
    logs = (result.stdout or "") + ("\n" if result.stdout and result.stderr else "") + (result.stderr or "")
    status = "✅ Completed" if result.returncode == 0 else "❌ Failed"
    status_class = "master-ok" if result.returncode == 0 else "master-bad"

    display(HTML(STYLE + f"""
    <div class="master-wrap">
      <div class="master-title">{status}: {label}</div>
      <p class="master-subtitle">Finished in {elapsed} · exit code {result.returncode}</p>
      {"<details><summary>Show logs</summary><div class='master-log'>" + _escape(logs[-6000:]) + "</div></details>" if logs.strip() else ""}
      <div class="master-badge {status_class}">{status}</div>
    </div>
    """))
    return result

def display_log(title, logs):
    display(HTML(STYLE + f"""
    <div class="master-wrap">
      <div class="master-title">📜 {_escape(title)}</div>
      <div class="master-log">{_escape(logs[-8000:])}</div>
    </div>
    """))

def port_open(host, port, timeout=1.5):
    try:
        with socket.create_connection((host, int(port)), timeout=timeout):
            return True
    except OSError:
        return False

def wait_for_port(port, host="127.0.0.1", timeout=60):
    deadline = time.time() + timeout
    while time.time() < deadline:
        if port_open(host, port):
            return True
        time.sleep(1)
    return False

def url_ok(url, timeout=2):
    try:
        urllib.request.urlopen(url, timeout=timeout)
        return True
    except Exception:
        return False

def pretty_size(path):
    p = Path(path)
    if not p.exists():
        return "not found"
    size = p.stat().st_size
    units = ["B", "KB", "MB", "GB", "TB"]
    idx = 0
    while size >= 1024 and idx < len(units) - 1:
        size /= 1024
        idx += 1
    return f"{size:.2f} {units[idx]}"


In [ ]:
#@title ⚙️ Mission Control
#@markdown Customize these only if you want a different model or port setup.

MODEL_NAME = "gemma4-uncensored" #@param {type:"string"}
MODEL_URL = "https://huggingface.co/HauhauCS/Gemma-4-E4B-Uncensored-HauhauCS-Aggressive/resolve/main/Gemma-4-E4B-Uncensored-HauhauCS-Aggressive-Q8_K_P.gguf" #@param {type:"string"}
MODEL_PATH = "/content/model.gguf" #@param {type:"string"}
OLLAMA_PORT = 11434 #@param {type:"integer"}
WEBUI_PORT = 8000 #@param {type:"integer"}

panel(
    "🧭 Mission Control",
    "Configuration locked in. Continue to the install cell.",
    cards=[
        ("Model name", MODEL_NAME),
        ("Model path", MODEL_PATH),
        ("Ollama API", f"localhost:{OLLAMA_PORT}"),
        ("Open WebUI", f"localhost:{WEBUI_PORT}"),
    ],
    badge="Ready for launch",
    badge_class="master-ok",
)


In [ ]:
#@title 📦 Step 1 · Install Dependencies
#@markdown Installs system packages, Ollama, and the Python packages required for Pinggy and Open WebUI.

commands = [
    ("System tools", "apt-get update -qq && apt-get install -y -qq lshw pciutils zstd lsof wget curl"),
    ("Ollama runtime", "curl -fsSL https://ollama.com/install.sh | sh"),
    ("Python packages", "python -m pip install -q --upgrade requests pinggy open-webui"),
]

all_ok = True
for label, command in commands:
    result = run_shell(label, command, timeout=1200)
    if result.returncode != 0:
        all_ok = False

if all_ok:
    success("All dependencies are installed. Continue to Step 2.")
else:
    failure("One or more dependencies failed to install. Open the logs above, fix the issue, then re-run this cell.")


In [ ]:
#@title 🚀 Step 2 · Start Ollama Server
#@markdown Starts the Ollama background server and verifies the API port.

import subprocess
import time
from pathlib import Path

OLLAMA_LOG = "/content/ollama.log"

# Show GPU diagnostics when available.
gpu = subprocess.run(
    "nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader 2>/dev/null || true",
    shell=True,
    capture_output=True,
    text=True,
).stdout.strip()

if gpu:
    panel(
        "🎮 GPU detected",
        "Colab GPU is visible to the runtime.",
        cards=[("GPU", gpu)],
        badge="GPU ready",
        badge_class="master-ok",
    )
else:
    warn("No GPU detected. In Colab, go to Runtime → Change runtime type → GPU, then restart and run again.")

if port_open("127.0.0.1", OLLAMA_PORT):
    success(f"Ollama already appears to be running on port {OLLAMA_PORT}.")
else:
    log_file = open(OLLAMA_LOG, "a")
    ollama_process = subprocess.Popen(
        ["ollama", "serve"],
        stdout=log_file,
        stderr=subprocess.STDOUT,
    )
    if wait_for_port(OLLAMA_PORT, timeout=60):
        success(f"Ollama is live on port {OLLAMA_PORT}.")
    else:
        failure(f"Ollama did not start on port {OLLAMA_PORT}. Showing the latest log:")
        if Path(OLLAMA_LOG).exists():
            display_log("Ollama log", Path(OLLAMA_LOG).read_text(errors="ignore"))


In [ ]:
#@title 📥 Step 3 · Download Model
#@markdown Downloads the GGUF model. Large models can take several minutes depending on Colab/network speed.

from pathlib import Path

Path(MODEL_PATH).parent.mkdir(parents=True, exist_ok=True)

existing_size = Path(MODEL_PATH).stat().st_size if Path(MODEL_PATH).exists() else 0
if existing_size > 1_000_000_000:
    success(f"Model already exists at {MODEL_PATH} ({pretty_size(MODEL_PATH)}). Skipping download.")
else:
    warn("This is a large download. Keep this tab open until the cell completes.")
    result = run_shell(
        "Download GGUF model",
        f"wget -c --progress=dot:giga -O {shlex.quote(MODEL_PATH)} {shlex.quote(MODEL_URL)}",
        timeout=None,
    )
    if result.returncode == 0 and Path(MODEL_PATH).exists():
        success(f"Model downloaded successfully: {pretty_size(MODEL_PATH)}")
    else:
        failure("Model download failed. Check the log above and verify the MODEL_URL.")


In [ ]:
#@title 🔧 Step 4 · Register Model with Ollama
#@markdown Creates an Ollama model entry from the downloaded GGUF file.

from pathlib import Path

if not Path(MODEL_PATH).exists():
    failure(f"Model file not found: {MODEL_PATH}. Run Step 3 first.")
else:
    modelfile_path = Path("/content/Modelfile")
    modelfile_path.write_text(f"FROM {MODEL_PATH}\n", encoding="utf-8")

    result = run_shell(
        f"Register {MODEL_NAME}",
        f"ollama create {shlex.quote(MODEL_NAME)} -f {shlex.quote(str(modelfile_path))}",
        timeout=1800,
    )

    if result.returncode == 0:
        success(f"Model '{MODEL_NAME}' is registered with Ollama.")
        run_shell("Available Ollama models", "ollama list", timeout=120)
    else:
        failure("Model registration failed. Make sure Ollama is running and the GGUF file is valid.")


In [ ]:
#@title 🔗 Step 5 · Open Tunnels
#@markdown Creates Pinggy tunnel URLs for the Ollama API and Open WebUI.

import time
from pathlib import Path

try:
    import pinggy
except Exception as e:
    failure(f"Could not import pinggy: {e}. Re-run Step 1.")
    raise

def get_tunnel_url(tunnel, retries=20, delay=1.5):
    for _ in range(retries):
        urls = getattr(tunnel, "urls", None)
        if urls:
            return urls[0]
        time.sleep(delay)
    return None

OLLAMA_TUNNEL_URL = None
WEBUI_TUNNEL_URL = None

try:
    tunnel_ollama = pinggy.start_tunnel(
        forwardto=f"localhost:{OLLAMA_PORT}",
        headermodification=[f"u:Host:localhost:{OLLAMA_PORT}"],
    )
    OLLAMA_TUNNEL_URL = get_tunnel_url(tunnel_ollama)
except Exception as e:
    failure(f"Ollama tunnel failed: {e}")

try:
    tunnel_webui = pinggy.start_tunnel(forwardto=f"localhost:{WEBUI_PORT}")
    WEBUI_TUNNEL_URL = get_tunnel_url(tunnel_webui)
except Exception as e:
    failure(f"Open WebUI tunnel failed: {e}")

links = []
if OLLAMA_TUNNEL_URL:
    links.append(("Ollama API", OLLAMA_TUNNEL_URL))
if WEBUI_TUNNEL_URL:
    links.append(("Open WebUI", WEBUI_TUNNEL_URL))

if links:
    Path("/content/tunnel_links.txt").write_text(
        "\n".join(f"{name}: {url}" for name, url in links),
        encoding="utf-8",
    )

    buttons = "".join(
        f'<a class="master-button" href="{_escape(url)}" target="_blank">Open {name}</a>'
        for name, url in links
    )
    display(HTML(STYLE + f"""
    <div class="master-wrap">
      <div class="master-title">🌉 Tunnels are ready</div>
      <p class="master-subtitle">Copy these links now. The WebUI link will start working after Step 6 launches Open WebUI.</p>
      <div class="master-grid">
        {''.join(f'<div class="master-card"><div class="master-kicker">{_escape(name)}</div><div class="master-value">{_escape(url)}</div></div>' for name, url in links)}
      </div>
      <div>{buttons}</div>
      <div class="master-badge master-ok">Links saved to /content/tunnel_links.txt</div>
    </div>
    """))
else:
    failure("No tunnel URLs were returned. Re-run this cell or check Pinggy availability.")


## ⚠️ Final Step Notes

Before running the final cell:

- Open the **Open WebUI** tunnel link from Step 5 after the final cell says it is ready.
- The first model response can take several minutes because the model has to load into memory.
- Do **not** stop the final cell while using Open WebUI.
- When finished, stop the Colab runtime so public tunnels close.


In [ ]:
#@title 🌐 Step 6 · Launch Open WebUI
#@markdown Keep this cell running. It starts the web server that powers the Open WebUI tunnel.

import os
import subprocess
import time
from pathlib import Path

WEBUI_LOG = "/content/open_webui.log"

env = os.environ.copy()
env["OLLAMA_BASE_URL"] = f"http://127.0.0.1:{OLLAMA_PORT}"

if port_open("127.0.0.1", WEBUI_PORT):
    success(f"Something is already listening on port {WEBUI_PORT}. Try opening your WebUI tunnel.")
else:
    log_file = open(WEBUI_LOG, "a")
    proc = subprocess.Popen(
        ["open-webui", "serve", "--host", "0.0.0.0", "--port", str(WEBUI_PORT)],
        stdout=log_file,
        stderr=subprocess.STDOUT,
        env=env,
    )

    if wait_for_port(WEBUI_PORT, timeout=90):
        webui_link = globals().get("WEBUI_TUNNEL_URL", "")
        open_button = (
            f'<a class="master-button" href="{_escape(webui_link)}" target="_blank">Open Open WebUI</a>'
            if webui_link else ""
        )
        display(HTML(STYLE + f"""
        <div class="master-wrap">
          <div class="master-title">🎉 Open WebUI is running</div>
          <p class="master-subtitle">Keep this cell running. Use the tunnel link from Step 5 to open the interface.</p>
          <div class="master-grid">
            <div class="master-card"><div class="master-kicker">Local URL</div><div class="master-value">http://127.0.0.1:{WEBUI_PORT}</div></div>
            <div class="master-card"><div class="master-kicker">Ollama Base URL</div><div class="master-value">{_escape(env["OLLAMA_BASE_URL"])}</div></div>
            <div class="master-card"><div class="master-kicker">Model</div><div class="master-value">{_escape(MODEL_NAME)}</div></div>
          </div>
          <div>{open_button}</div>
          <div class="master-badge master-ok">Ready to chat</div>
        </div>
        """))
    else:
        failure("Open WebUI did not start in time. Showing the latest log:")
        if Path(WEBUI_LOG).exists():
            display_log("Open WebUI log", Path(WEBUI_LOG).read_text(errors="ignore"))

    # Keep the process alive so the tunnel keeps working.
    proc.wait()


## 🛠️ Troubleshooting

**Open WebUI link does not load:**  
Make sure Step 6 is still running, then refresh the tunnel URL from Step 5.

**Ollama model is missing inside WebUI:**  
Run Step 4 again, then check `ollama list`.

**First response is slow:**  
That is normal. The model loads into memory on the first request.

**Colab disconnects or runtime restarts:**  
Run all cells again from the top. Colab storage is temporary unless you mount Drive.
